In [3]:
pip install geopandas

   ---------------------------------------- 0.0/22.9 MB ? eta -:--:--
   ---------- ----------------------------- 6.0/22.9 MB 30.8 MB/s eta 0:00:01
   ------------------ --------------------- 10.7/22.9 MB 32.0 MB/s eta 0:00:01
   -------------------------- ------------- 15.5/22.9 MB 25.0 MB/s eta 0:00:01
   ------------------------------------- -- 21.8/22.9 MB 26.0 MB/s eta 0:00:01
   ---------------------------------------- 22.9/22.9 MB 24.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ------------------------------- -------- 5.0/6.3 MB 23.2 MB/s eta 0:00:01
   ---------------------------------------- 6.3/6.3 MB 21.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 23.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import os

# 1. CONFIGURACIÓ DE RUTES
ruta_mapa_seccions = '../data/raw/BarcelonaCiutat_SeccionsCensals.csv' # El teu nou fitxer!
carpeta_indicadors = '../data/raw/indicadors_crb/'             # On tinguis els GeoJSON del CRB
ruta_sortida = '../data/processed/02_dades_CRB_SeccioCensal.xlsx'

print("🛰️ Carregant el mapa base de seccions censals des del CSV...")
df_seccions = pd.read_csv(ruta_mapa_seccions)

# 2. FABRIQUEM EL CODI_UNIC EXACTE (Prefix BCN + Districte + Secció)
districte_str = df_seccions['codi_districte'].astype(str).str.zfill(2)
seccio_str = df_seccions['codi_seccio_censal'].astype(str).str.zfill(3)

# AFEGIM EL PREFIX DIRECTAMENT A L'ARREL
df_seccions['CODI_UNIC'] = '08019' + districte_str + seccio_str

# 3. CONVERTIM EL TEXT A MAPA REAL (WKT a Geometria)
# Agafem la columna 'geometria_wgs84' i la transformem en polígons espacials
df_seccions['geometry'] = df_seccions['geometria_wgs84'].apply(wkt.loads)

# Creem el mapa oficial de Geopandas (indiquem que està en format GPS estàndard EPSG:4326)
seccions = gpd.GeoDataFrame(df_seccions[['CODI_UNIC', 'geometry']], geometry='geometry', crs="EPSG:4326")
# TRUC D'ENGINYER: Projectem el mapa a UTM 31N (EPSG:25831) per treballar en metres i calcular centres perfectes
seccions = seccions.to_crs(epsg=25831)

# =====================================================================
# 4. BUCLE DE PROCESSAMENT D'INDICADORS DEL CRB (SPATIAL JOIN)
# =====================================================================
master_df = pd.DataFrame(columns=['CODI_UNIC'])

print("\n🔍 Buscant indicadors GeoJSON del CRB a la carpeta...")
fitxers = [f for f in os.listdir(carpeta_indicadors) if f.endswith('.geojson') or f.endswith('.json')]

for fitxer in fitxers:
    nom_indicador = fitxer.split('.')[0] # Treiem l'extensió per tenir el nom net
    print(f"🔄 Processant indicador: {nom_indicador}...")
    
    # Carreguem els edificis d'aquest indicador
    edificis = gpd.read_file(os.path.join(carpeta_indicadors, fitxer))
    
    # Assegurar que els sistemes de coordenades coincideixen
    if edificis.crs != seccions.crs:
        edificis = edificis.to_crs(seccions.crs)
    
    # Convertim els edificis a un sol punt (centroide) per evitar solapaments
    edificis['geometry'] = edificis.geometry.centroid
    
    # UNIÓN ESPACIAL: Fiquem els edificis dins dels barris
    join = gpd.sjoin(edificis, seccions, how="inner", predicate="within")
    
    # Agrupem per CODI_UNIC i fem la mitjana (la variable del CRB es diu 'kpis')
    # NOTA: Si en algun arxiu la columna no es diu 'kpis', avisa'm!
    resum_seccio = join.groupby('CODI_UNIC')['kpis'].mean().reset_index()
    resum_seccio.rename(columns={'kpis': nom_indicador}, inplace=True)
    
    # Ajuntem al Master DataFrame
    if master_df.empty:
        master_df = resum_seccio
    else:
        master_df = pd.merge(master_df, resum_seccio, on='CODI_UNIC', how='outer')

# =====================================================================
# 5. EXPORTACIÓ FINAL A L'EXCEL
# =====================================================================
print(f"\n💾 Guardant resultats a {ruta_sortida}...")
os.makedirs(os.path.dirname(ruta_sortida), exist_ok=True)
master_df.to_excel(ruta_sortida, index=False)

print("✅ Procés finalitzat amb èxit!")
print(f"S'han integrat {len(fitxers)} indicadors per a {len(master_df)} seccions censals de Barcelona.")

🛰️ Carregant el mapa base de seccions censals des del CSV...

🔍 Buscant indicadors GeoJSON del CRB a la carpeta...
🔄 Processant indicador: annual_net_incomes_household_per_building__2022-01-01...
🔄 Processant indicador: average_dwelling_area__2025-05-14...
🔄 Processant indicador: CVI__2023-01-01...
🔄 Processant indicador: heating_degree_days__2024-01-01...
🔄 Processant indicador: heating_thermal_demand_intensity__2025-03-06...
🔄 Processant indicador: percentage_population_over_65__2022-01-01...
🔄 Processant indicador: percentage_single_person_households__2022-01-01...
🔄 Processant indicador: torrid_nights__2024-01-01...
🔄 Processant indicador: vegetation_index_avg__2022-01-01...

💾 Guardant resultats a ../data/processed/02_dades_CRB_SeccioCensal.xlsx...
✅ Procés finalitzat amb èxit!
S'han integrat 9 indicadors per a 1068 seccions censals de Barcelona.


In [10]:
import geopandas as gpd
import os

# 1. RUTA DE LA CARPETA
carpeta_indicadors = '../data/raw/indicadors_crb/'

print("📊 INFORME DE VOLUMETRIA D'ARXIUS (SANITY CHECK) 📊\n")

# Busquem tots els arxius de la carpeta
fitxers = [f for f in os.listdir(carpeta_indicadors) if f.endswith('.geojson') or f.endswith('.json')]

if len(fitxers) == 0:
    print("❌ No s'ha trobat cap arxiu a la carpeta indicada.")
else:
    print(f"S'han trobat {len(fitxers)} arxius. Calculant...\n")

    for fitxer in fitxers:
        ruta = os.path.join(carpeta_indicadors, fitxer)
        
        # Llegim el mapa a la memòria
        geo_df = gpd.read_file(ruta)
        num_edificis = len(geo_df)
        
        # Lògica d'enginyer per detectar el problema del servidor web
        if num_edificis in [5000, 10000] or num_edificis % 1000 == 0:
            estat = "⚠️ ALERTA: Límit de seguretat del servidor detectat!"
        elif num_edificis > 65000:
            estat = "✅ OK: Sembla un arxiu complet de tota Barcelona."
        else:
            estat = "❓ INCOMPLET: Possible descàrrega parcial segons el zoom."
            
        print(f"📄 {fitxer:<35} | {num_edificis:>6} edificis | {estat}")

print("\n🔍 Anàlisi finalitzat.")

📊 INFORME DE VOLUMETRIA D'ARXIUS (SANITY CHECK) 📊

S'han trobat 9 arxius. Calculant...

📄 annual_net_incomes_household_per_building__2022-01-01.geojson |  61578 edificis | ❓ INCOMPLET: Possible descàrrega parcial segons el zoom.
📄 average_dwelling_area__2025-05-14.geojson |  61578 edificis | ❓ INCOMPLET: Possible descàrrega parcial segons el zoom.
📄 CVI__2023-01-01.geojson             |  61578 edificis | ❓ INCOMPLET: Possible descàrrega parcial segons el zoom.
📄 heating_degree_days__2024-01-01.geojson |  61578 edificis | ❓ INCOMPLET: Possible descàrrega parcial segons el zoom.
📄 heating_thermal_demand_intensity__2025-03-06.geojson |  61578 edificis | ❓ INCOMPLET: Possible descàrrega parcial segons el zoom.
📄 percentage_population_over_65__2022-01-01.geojson |  61578 edificis | ❓ INCOMPLET: Possible descàrrega parcial segons el zoom.
📄 percentage_single_person_households__2022-01-01.geojson |  61578 edificis | ❓ INCOMPLET: Possible descàrrega parcial segons el zoom.
📄 torrid_nights__202